<h2> Search Evaluation </h2>

Now that we have ground truth data, we can evaluate how well our search retrieves the correct documents.

For each question in our ground truth dataset, we run search. Then we check whether the results include the correct document.

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [4]:
q = ground_truth[0]
q

{'question': 'Can I still join the course if I found it late?',
 'document': '74eb249bbf'}

In [6]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [8]:
for d in results:
    print(f'{d["doc_id"]} == {doc_id}: {d["doc_id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
69d122f12e == 74eb249bbf: False


In [9]:
relevance = []

for d in results:
    relevance.append(int(d["doc_id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [12]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [13]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Can I still join the course if I found it late?


[1, 0, 0, 0, 0]

In [14]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where is the live stream link for Office Hours usually posted before the session starts?


[1, 0, 0, 0, 0]

In [15]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [18]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [19]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [21]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [22]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [27]:
relevance_sample = compute_relevance_total(ground_truth_sample, text_search)
relevance_sample

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [24]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

In [25]:
relevance_total

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0,

<h2> Evaluation Metrics - Hit Rate | MRR </h2>

<h3> Hit Rate  </h3> Hit Rate (also called Recall@k) measures the fraction of queries where the correct document appears anywhere in the results

In [28]:
cnt = 0

for line in relevance_sample:
    if 1 in line:
        cnt = cnt + 1

cnt

10

In [31]:
hit_rate_sample  = cnt / len(relevance_sample)
hit_rate_sample
# 0.666

0.6666666666666666

In [32]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [33]:
hit_rate(relevance_sample)

0.6666666666666666

<h2> MRR Mean Reciprocal Rank</h2>

Hit Rate tells us if we found the right document, but not where it was.

MRR also considers the position.

In [34]:
total_score = 0.0
# We use rank + 1 because Python counts positions from zero. 
# The first position should score 1/1, and without the + 1 we'd divide by zero.
for line in relevance_sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

8.5

MRR is the average of these scores across all queries. It rewards systems that put the correct document near the top.

Hit Rate is the upper bound for MRR. In practice, MRR is usually smaller because some correct documents are found below the first position.

In [35]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [36]:
mrr(relevance_sample)
# 0.822

0.5666666666666667

In [37]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

A few things to keep in mind when reading these numbers:

Our ground truth assumes only one relevant document per query. In practice, other retrieved documents might also be relevant. A 50% hit rate does not mean that half the results are useless. It means the document we generated the question from did not appear in the top results for half the queries. Other relevant documents may still be there.

With synthetic data, the generated questions can be too close to the original FAQ text. This inflates hit rate and MRR. If you see numbers above 95%, treat them with caution and check whether the questions are realistic enough.

Good thresholds depend on your use case. A 50% hit rate is acceptable for some applications, while others need 90% or higher. The right number depends on how much the downstream LLM can compensate for imperfect retrieval. It also depends on user tolerance for wrong answers.

Look at the system holistically. A high MRR means the relevant document is near the top, which helps the LLM focus on the right context. A low MRR with a high hit rate means the document is there, but buried under less relevant results.

<h2> Search Tuning </h2>

In [38]:
#  Boost only the question match
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [39]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8973451327433628, 'mrr': 0.8020058997050146}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8938053097345132, 'mrr': 0.7782005899705012}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8407079646017699, 'mrr': 0.7229203539823008}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8247787610619469, 'mrr': 0.6930678466076693}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7911504424778761, 'mrr': 0.6583775811209438}


In [40]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [41]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

In [42]:
df_results = pd.DataFrame(results)
# Sort by MRR
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.978761,0.870088
6,1.0,4.0,0.1,0.978761,0.869233
23,2.0,10.0,0.5,0.971681,0.868555
18,2.0,4.0,0.1,0.973451,0.866401
4,1.0,2.0,0.2,0.976991,0.865162
19,2.0,4.0,0.2,0.973451,0.864867
35,5.0,10.0,0.5,0.973451,0.864867
3,1.0,2.0,0.1,0.973451,0.864867
8,1.0,4.0,0.5,0.969912,0.864867
22,2.0,10.0,0.2,0.973451,0.864838


In [43]:
# the weights are given based on when there are only 79 questions in FAQ 
# but when I ran for the firs time it has already increased til 113 and 
# hence some of the results and sorting has a bit off and hence the given weights may not fully capture the truth.
# as it is mainly for learning basis, i will use the given weights for learning purpose
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Hit Rate tells us whether the correct document appears at all. MRR tells us whether it appears near the top. A document near the top is more likely to be used by the RAG prompt.

Tuning Workflow
Search parameters can look arbitrary. This includes field boosts, number of results, filters, and other settings. Evaluation gives us a way to compare settings with evidence.

Grid search is fine when there are only a few settings. For a larger parameter space, use a smarter search strategy. You can sample random combinations, use Bayesian optimization, or keep a validation split so you don't overfit the evaluation set.

For text search on our dataset, grid search takes about one second per combination. That makes it practical to try many options. When each evaluation takes minutes instead of seconds, grid search becomes too expensive. In those cases, use Bayesian optimization with a library like hyperopt. It explores the parameter space more efficiently by focusing on combinations that are likely to improve the metric.

We return 5 results from search. Increasing top-K to 10 would improve hit rate because there are more chances to find the correct document. But more results means more context sent to the LLM. That costs more and makes it harder for the model to identify what is relevant. Five results is a reasonable default for short FAQ-style documents.

Next, we'll move from retrieval quality to answer quality and evaluate the full RAG pipeline.